# <img src="playwright-logo.svg" height="50" style="vertical-align:middle; margin-right:12px"> Playwright

### Modern, reliable end-to-end browser automation

> **End-to-end (E2E) browser automation** = driving a *real* browser through a
> complete user journey - open a page, click, type, navigate, wait for results -
> exactly as a human would, to **test** or **automate** an app from the outside in.

**Agenda (~30 min)**

1. What is Playwright?
2. Playwright vs. Selenium vs. Puppeteer (vs. Cypress)
3. Supported languages & browsers
4. Playwright + **Python** (our stack)
5. Output formats: screenshots, PDF, video, traces
6. Interacting with JavaScript
7. Headless browsers in practice
8. Tooling extras: codegen, trace viewer
9. Q&A

## 1. What is Playwright?

* Open-source browser automation framework by **Microsoft** (first release: 2020)
* Built by the **original Puppeteer team** after moving from Google to Microsoft
* Drives real browsers - **Chromium, Firefox, WebKit** - with a single API
* Primary use cases:
  * **End-to-end testing** of web apps
  * **Web scraping** / data extraction
  * **Automation** of repetitive browser tasks (form filling, report downloads…)
  * **Rendering**: screenshots & PDFs of pages

### Why people like it
* **Auto-waiting**: actions wait for elements to be ready -> far fewer flaky `sleep()` hacks
* **One API, three browser engines** (incl. WebKit ≈ Safari)
* Fast: speaks the browser's native protocols (CDP & friends) instead of WebDriver HTTP
* Batteries included: network interception, tracing, video, device emulation


## 2. Competitive landscape

| | **Playwright** | **Selenium** | **Puppeteer** | **Cypress** |
|---|---|---|---|---|
| Maintainer | Microsoft | SeleniumHQ (OSS) | Google | Cypress.io |
| First release | 2020 | 2004 | 2017 | 2017 |
| Browsers | Chromium, Firefox, WebKit | All major (via WebDriver) | Chromium (Firefox experimental) | Chromium-family, Firefox, WebKit (exp.) |
| Languages | Python, JS/TS, Java, .NET | Python, Java, C#, Ruby, JS, Kotlin… | JavaScript/TypeScript only | JavaScript/TypeScript only |
| Protocol | Native (CDP etc.), websocket | WebDriver (HTTP) -> slower | CDP | Runs *inside* the browser |
| Auto-waiting | ✅ built-in | ❌ manual waits | partial | ✅ |
| Ecosystem age | newer, growing fast | huge, mature, many integrations | large (JS world) | large (JS testing) |


### When would you pick which?

* **Selenium** - legacy suites, exotic browsers/grids, broadest language support, mature enterprise tooling
* **Puppeteer** - JS-only shops focused on Chrome automation/scraping
* **Cypress** - JS-only, developer-centric in-browser testing experience
* **Playwright** - cross-browser, multi-language, modern API, great Python support -> **our pick** ✅


## 3. Languages & browsers

**Official language bindings**

* Python 🐍 (sync **and** async API)
* JavaScript / TypeScript (the "reference" implementation, incl. the `@playwright/test` runner)
* Java
* .NET (C#)

**Browser engines** (Playwright ships its own patched builds)

* **Chromium** -> Chrome, Edge (can also drive branded Chrome/Edge installs)
* **Firefox**
* **WebKit** -> the engine behind Safari (huge plus: test "Safari" on Linux/Windows!)

**Plus**

* Mobile **device emulation** (viewport, user agent, touch, geolocation) - e.g. "iPhone 14"
* Runs on Linux, macOS, Windows, Docker, CI (GitHub Actions, GitLab CI, Jenkins…)


## 4. Playwright + Python

Two flavours:

```python
from playwright.sync_api import sync_playwright   # scripts, pytest
from playwright.async_api import async_playwright # asyncio apps, Jupyter ✅
```

* `pytest-playwright` plugin -> fixtures (`page`, `context`, `browser`), parallelism, retries
* In **Jupyter** we must use the async API (event loop already running)


In [ ]:
from playwright.async_api import async_playwright

pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=True)
page = await browser.new_page()

await page.goto("https://playwright.dev")
print("Title:", await page.title())

Title: Fast and reliable end-to-end testing for modern web apps | Playwright


In [ ]:
# Locators: the modern way to find elements (auto-waiting built in)
await page.get_by_role("link", name="Get started").click()
print("Now at:", page.url)

# Other locator styles:
# page.locator("css=button.submit")     CSS
# page.locator("xpath=//button")        XPath
# page.get_by_text("Welcome")           visible text
# page.get_by_label("Email")            form label
# page.get_by_test_id("checkout")       data-testid

Now at: https://playwright.dev/docs/intro


### Auto-waiting in one sentence

`await page.click(...)` automatically waits until the element is **attached, visible,
stable, enabled and not obscured** - no `time.sleep(3)` sprinkled everywhere. 🎉


## 5. Output formats

| Output | API | Notes |
|---|---|---|
| **Screenshot (PNG/JPEG)** | `page.screenshot()` | full page, viewport, or single element |
| **PDF** | `page.pdf()` | **Chromium + headless only**; paper size, margins, header/footer |
| **Video (WebM)** | `browser.new_context(record_video_dir=...)` | one video per page |
| **Trace (ZIP)** | `context.tracing.start()/stop()` | full timeline: screenshots, DOM snapshots, network - open in Trace Viewer |
| **HAR (network log)** | `record_har_path=...` | every request/response, replayable |
| Raw data | your code | scraped text, JSON, downloads (`page.expect_download()`) |


In [ ]:
# Screenshots
await page.goto("https://playwright.dev")
await page.screenshot(path="homepage.png")                      # viewport
await page.screenshot(path="homepage_full.png", full_page=True) # whole page
await page.get_by_role("navigation").first.screenshot(path="navbar.png")  # one element
print("Screenshots saved ✔")

In [ ]:
# PDF - Chromium, headless only
await page.emulate_media(media="print")
await page.pdf(path="playwright_dev.pdf", format="A4",
               margin={"top": "1cm", "bottom": "1cm"})
print("PDF saved ✔")

In [ ]:
# Video + trace are configured on the *context*
ctx = await browser.new_context(record_video_dir="videos/")
await ctx.tracing.start(screenshots=True, snapshots=True)

p2 = await ctx.new_page()
await p2.goto("https://playwright.dev")
await p2.get_by_role("link", name="Get started").click()

await ctx.tracing.stop(path="trace.zip")   # open with: playwright show-trace trace.zip
await ctx.close()                          # finalizes the video file
print("Video + trace saved ✔")

### 🇨🇭 Real-world demo: map.geo.admin.ch

Record a **video + trace** of a user panning and zooming the Swiss federal map.

* Maps are perfect for this: pure canvas, lots of async tile loading, no simple DOM to assert on
* We drive the map like a human: **mouse drag** to pan, **scroll wheel** to zoom
* The trace captures every tile request in the **network tab** - great talking point


In [ ]:
# Video + trace of panning & zooming map.geo.admin.ch
geo_ctx = await browser.new_context(
    viewport={"width": 1280, "height": 800},
    record_video_dir="videos/",
    record_video_size={"width": 1280, "height": 800},
    locale="de-CH",
)
await geo_ctx.tracing.start(screenshots=True, snapshots=True)

geo = await geo_ctx.new_page()
await geo.goto("https://map.geo.admin.ch", wait_until="load")
await geo.wait_for_timeout(5000)          # let the initial tiles render

# Dismiss a possible disclaimer/cookie popup (ignore if none appears)
for label in ["Schliessen", "Close", "OK", "Akzeptieren"]:
    try:
        await geo.get_by_role("button", name=label).click(timeout=1500)
        break
    except Exception:
        pass

cx, cy = 640, 400                          # centre of the viewport

# --- Pan: press, drag, release (steps=30 -> smooth motion in the video) ---
await geo.mouse.move(cx, cy)
await geo.mouse.down()
await geo.mouse.move(cx - 350, cy - 150, steps=30)
await geo.mouse.up()
await geo.wait_for_timeout(2000)           # tiles load for the new area

# --- Pan back the other way ---
await geo.mouse.move(cx, cy)
await geo.mouse.down()
await geo.mouse.move(cx + 250, cy + 200, steps=30)
await geo.mouse.up()
await geo.wait_for_timeout(2000)

# --- Zoom in: 3 scroll-wheel ticks over the map centre ---
await geo.mouse.move(cx, cy)
for _ in range(3):
    await geo.mouse.wheel(0, -400)         # negative deltaY = zoom in
    await geo.wait_for_timeout(1200)       # let each zoom level render

# --- Zoom out one step ---
await geo.mouse.wheel(0, 400)
await geo.wait_for_timeout(2000)

await geo.screenshot(path="geoadmin_final.png")
await geo_ctx.tracing.stop(path="geoadmin_trace.zip")

video_handle = geo.video                   # grab the handle *before* closing
await geo_ctx.close()                      # closing the context finalizes the .webm
print("Video :", await video_handle.path())
print("Trace : geoadmin_trace.zip   ->  playwright show-trace geoadmin_trace.zip")

**Things to point out when you replay this:**

* In the **video**: the drag and the stepwise zoom look exactly like a human session
* In the **trace**: open the Network tab -> watch the burst of WMTS tile requests after every pan/zoom
* No locators were used - `mouse.*` works on **canvas-based apps** (maps, charts, games) where there is no DOM to click


### Viewing traces: the Trace Viewer

```bash
playwright show-trace trace.zip
```

* **Film-strip timeline** of the whole run
* Every action with **before/after DOM snapshots** - real DOM, inspectable with devtools
* Network tab, console logs, and the exact source line per action
* No install needed? Drag the zip onto **https://trace.playwright.dev** (processed locally, nothing uploaded)
* Real-world workflow: `pytest --tracing retain-on-failure` -> time-travel debug CI failures


In [ ]:
# Open the Trace Viewer without blocking the notebook
import subprocess
subprocess.Popen(["playwright", "show-trace", "trace.zip"])

## 6. Interacting with JavaScript

Playwright can run arbitrary JS **inside the page** and pass data both ways.

* `page.evaluate(js, arg)` -> run JS, get back a **serializable** result
* `page.evaluate_handle(js)` -> get a **handle** to a live JS object/DOM node
* `page.add_init_script(js)` -> inject JS **before** any page script runs (great for stubbing)
* `page.expose_function(name, py_fn)` -> call **Python from the page's JS** 🤯
* `page.wait_for_function(js)` -> wait until a JS expression becomes truthy


In [ ]:
# Run JS in the page and get results back as Python objects
dims = await page.evaluate("""() => ({
    width:  window.innerWidth,
    height: window.innerHeight,
    links:  document.querySelectorAll('a').length,
})""")
print(dims)            # a real Python dict

In [ ]:
# Pass arguments from Python into the page
await page.evaluate(
    "([color, size]) => { document.body.style.background = color; document.body.style.fontSize = size; }",
    ["#ffeecc", "18px"],
)

# Call Python *from* the browser
async def py_hash(text):
    import hashlib
    return hashlib.sha256(text.encode()).hexdigest()[:12]

await page.expose_function("pyHash", py_hash)
print(await page.evaluate("() => window.pyHash('hello from the browser')"))

In [ ]:
# Inject JS before any page code runs - e.g. freeze time, stub APIs
ctx2 = await browser.new_context()
await ctx2.add_init_script("Date.now = () => 1735689600000;  // 2025-01-01")
p3 = await ctx2.new_page()
await p3.goto("https://playwright.dev")
print("Page thinks 'now' is:", await p3.evaluate("new Date(Date.now()).toISOString()"))
await ctx2.close()

## 7. Headless browsers

**Headless = browser runs without a visible UI window.**

* Default in Playwright (`headless=True`) -> ideal for **servers, Docker, CI**
* Faster, less memory, no display needed (no X server / no `xvfb` gymnastics)
* `headless=False` ("headed") -> watch what happens; great for **developing & debugging**
* `slow_mo=500` -> slow every action down by 500 ms so humans can follow
* Modern Chromium "new headless" mode renders almost identically to headed Chrome
  -> screenshots/PDFs match what users see

```python
browser = await pw.chromium.launch(
    headless=False,   # show the window (local dev)
    slow_mo=500,      # slow-motion for demos
)
```

**Debugging toolbox:** `PWDEBUG=1` env var -> opens the **Playwright Inspector**
(step through actions, pick locators live).


In [ ]:
# Typical CI-style headless run, condensed:
async with async_playwright() as p:
    b = await p.chromium.launch(headless=True)
    pg = await b.new_page()
    await pg.goto("https://example.com")
    print(await pg.title())
    await b.close()

### Debugging live: the Playwright Inspector

**Option 1 - `page.pause()` (one line, on demand)**

```python
# needs a headed browser
debug_browser = await pw.chromium.launch(headless=False)
dpage = await debug_browser.new_page()
await dpage.goto("https://playwright.dev")

await dpage.pause()      # ← Inspector window opens HERE
```

* Cell blocks (`[*]`) until you hit **Resume ▶** in the Inspector - not a hang!
* **Pick locator**: hover elements in the live browser, get the suggested locator
* Type locators in the Inspector -> matching elements highlight in the page

**Option 2 - `PWDEBUG=1` (step through *every* action)**

```python
import os
os.environ["PWDEBUG"] = "1"     # BEFORE launch! Forces headed + timeout 0
dbg = await pw.chromium.launch()
# ...later: del os.environ["PWDEBUG"]  (and relaunch)
```

> Rule of thumb: `page.pause()` for targeted debugging, `PWDEBUG=1` to watch a whole flow.


### What can you do in the Inspector?

It's a debugger for **Playwright actions**, not for Python itself.

* **Step through actions** - Resume ▶ / Step over ⤵ jump from one Playwright call
  to the next (`goto` -> `click` -> `fill`); plain Python in between just runs
* **Source pane** - highlights the line of the current action
  *(reliable in `.py` scripts; can be patchy from Jupyter cells)*
* **Action log + actionability checks** - per action: duration and *what it waited for*
  (attached, visible, stable, enabled, not obscured) -> find out **why** a step timed out
* **Pick locator** 🎯 - hover any element in the live browser, get the recommended
  locator (`get_by_role("button", name="Submit")`) - copy & paste into your code
* **Locator playground** - type a locator, matching elements highlight live in the page
* **Page stays interactive** - log in, dismiss popups, scroll manually… then Resume

> **Demo idea:** `pause()` -> *Pick locator* -> hover elements -> paste the suggested
> locator into a cell and run it. Connects straight back to the locators slide.


## 8. Tooling extras worth showing

* **Codegen** - record clicks and get Python code generated live:
  ```bash
  playwright codegen https://your-app.example
  ```
* **Trace Viewer** - time-travel debugging of a recorded run:
  ```bash
  playwright show-trace trace.zip
  ```
* **Network interception** - mock APIs, block trackers:
  ```python
  await page.route("**/*.png", lambda route: route.abort())
  await page.route("**/api/user", lambda route: route.fulfill(json={"name": "Demo"}))
  ```
* **Device emulation**:
  ```python
  iphone = p.devices["iPhone 14"]
  ctx = await browser.new_context(**iphone)
  ```
* **Storage state** - log in once, reuse the session across tests


In [ ]:
# Clean up
await browser.close()
await pw.stop()
print("Done ✅")

## 9. Wrap-up

* Playwright = modern, cross-browser, multi-language automation - excellent Python support
* Beats Selenium on speed & ergonomics, beats Puppeteer on browser/language coverage
* Outputs: screenshots, PDFs, videos, traces, HAR - plus anything you scrape
* Deep JS interop: `evaluate`, init scripts, even calling Python from the page
* Headless by default -> CI/Docker friendly; headed + Inspector for debugging

**Resources**

* Docs (Python): https://playwright.dev/python
* GitHub: https://github.com/microsoft/playwright-python
* Trace Viewer demo: https://trace.playwright.dev

### Questions? 🙋
